# A Multidimensional Predictive and Clustering Approach to Understanding State-Level Chronic Disease Disparities

## Team 8 - CMPE 255 Spring 2026

**Team Members:**
- Bharath Kumar A [018221268]
- Mithil Sri Sai Devisetty [019153394]
- Mani Mokshith Noonety [019100458]
- Tej Kiran Yenugunti [019104878]

---

## Project Overview

This notebook implements a comprehensive data-driven pipeline to analyze chronic disease disparities across U.S. states. We integrate eight public health datasets and apply machine learning techniques to identify predictive factors and state archetypes.

## 1. Setup and Library Installation

In [ ]:
# Install gdown for Google Drive data downloads
!pip install gdown --quiet

## 2. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data downloading
import gdown

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Advanced ML
from xgboost import XGBRegressor

# Statistical analysis
from scipy import stats

# Settings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 3. Data Loading

We will load eight state-level datasets from public repositories.

### 3.1 Dataset 1: Chronic Disease (CDC BRFSS)

Contains state-level chronic disease metrics by year, topic, and gender.

In [ ]:
# Download Chronic Disease Dataset
print("Downloading Chronic Disease dataset...")
gdown_id = "1GA_camLQGvBYLLnPa9vcNQvfZHWBWJhN"
gdown.download(id=gdown_id, output="../data/ChronicDisease.csv", quiet=False)

# Load the dataset
df_chronic = pd.read_csv("../data/ChronicDisease.csv")

print(f"\n✓ Loaded Chronic Disease dataset: {df_chronic.shape}")
print(f"Columns: {df_chronic.shape[1]}")
print(f"Rows: {df_chronic.shape[0]}")

In [ ]:
# Initial exploration
print("Dataset Info:")
df_chronic.info()

print("\nFirst 5 rows:")
df_chronic.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
df_chronic.isnull().sum()

### 3.2 Clean Chronic Disease Dataset

We'll clean the dataset by:
1. Removing unnecessary columns
2. Filtering relevant disease topics
3. Removing non-state entries (US aggregates, territories)
4. Handling missing values

In [ ]:
# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Response", "DataValueAlt", "DataValueFootnoteSymbol", "DataValueFootnote",
    "LowConfidenceLimit", "HighConfidenceLimit",
    "StratificationCategory2", "Stratification2",
    "StratificationCategory3", "Stratification3",
    "StratificationCategoryID2", "StratificationID2",
    "StratificationCategoryID3", "StratificationID3",
    "ResponseID", "Geolocation", "LocationID", "TopicID", "QuestionID", "DataValueTypeID"
]

df_chronic_clean = df_chronic.drop(columns=[c for c in cols_to_drop if c in df_chronic.columns])

print(f"✓ Dropped {len([c for c in cols_to_drop if c in df_chronic.columns])} unnecessary columns")
print(f"Remaining columns: {df_chronic_clean.shape[1]}")

In [ ]:
# Step 2: Filter relevant disease topics (remove non-chronic disease topics)
topics_to_drop = [
    "Nutrition, Physical Activity, and Weight Status",
    "Health Status",
    "Social Determinants of Health",
    "Immunization",
    "Cognitive Health and Caregiving",
    "Maternal Health"
]

df_chronic_clean = df_chronic_clean[~df_chronic_clean["Topic"].isin(topics_to_drop)]

print(f"✓ Filtered to chronic disease topics only")
print(f"Remaining rows: {df_chronic_clean.shape[0]}")

In [ ]:
# Step 3: Remove rows with missing DataValue
df_chronic_clean = df_chronic_clean[df_chronic_clean["DataValue"].notna()]

print(f"✓ Removed rows with missing disease values")
print(f"Remaining rows: {df_chronic_clean.shape[0]}")

In [ ]:
# Step 4: Standardize data types and rename columns
df_chronic_clean["YearStart"] = df_chronic_clean["YearStart"].astype(int)
df_chronic_clean["YearEnd"] = df_chronic_clean["YearEnd"].astype(int)
df_chronic_clean["DataValue"] = pd.to_numeric(df_chronic_clean["DataValue"], errors="coerce")
df_chronic_clean["Year"] = df_chronic_clean["YearStart"]
df_chronic_clean = df_chronic_clean.rename(columns={"LocationDesc": "State"})

print("✓ Standardized data types and renamed columns")
print(f"\\nData types:")\nprint(df_chronic_clean.dtypes)

In [ ]:
# Step 5: Remove non-state entries (US aggregates, territories)
df_chronic_clean = df_chronic_clean[
    (df_chronic_clean["State"] != "United States") & 
    (df_chronic_clean["State"] != "Puerto Rico")
].reset_index(drop=True)

print(f"✓ Removed non-state entries")
print(f"Final cleaned dataset shape: {df_chronic_clean.shape}")
print(f"\\nUnique states: {df_chronic_clean['State'].nunique()}")
print(f"Unique disease topics: {df_chronic_clean['Topic'].nunique()}")

In [ ]:
# Display cleaned data sample
print("Cleaned Chronic Disease Dataset Sample:")
df_chronic_clean.head(10)

## 4. Healthcare Expenditure Data (2015-2020)

Loading state-level healthcare spending data to analyze resource allocation patterns.

In [ ]:
# Download Healthcare Expenditure Dataset
print("Downloading Healthcare Expenditure dataset...")
gdown.download(id="1fcDO9LkieCtt5C1jr5ejdHe9_MsFR0pU", output="../data/Healthexpense.csv", quiet=False)

# Load dataset
df_expenditure = pd.read_csv("../data/Healthexpense.csv")

print(f"\n✓ Loaded Healthcare Expenditure dataset: {df_expenditure.shape}")
print(f"Years covered: 2015-2020")
df_expenditure.head()

In [ ]:
# Check data structure
print("Expenditure dataset info:")
df_expenditure.info()

print("\nColumn names:")
print(df_expenditure.columns.tolist())

In [ ]:
# Calculate total expenditure 2015-2020
# Note: Need to identify year columns and aggregate
print("TODO: Aggregate spending across 2015-2020")
print("TODO: Handle missing values")
print("TODO: Standardize state names for merging")

# Initial aggregation attempt
if 'Y2015' in df_expenditure.columns:
    year_cols = ['Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019', 'Y2020']
    df_expenditure['Total_Expense_2015_2020'] = df_expenditure[year_cols].sum(axis=1)
    print("\n✓ Calculated total expenditure")
else:
    print("\n⚠ Need to identify year columns correctly")

## 5. Healthcare Workforce Data

Loading physicians and hospitals datasets to analyze healthcare capacity.

In [ ]:
# Download Physicians Dataset
print("Downloading Physicians dataset...")
gdown.download(id="1giEZL5n4DP3G4WaVt-v7_rqdKO6m2dNa", output="../data/Physicians.csv", quiet=False)

df_physicians = pd.read_csv("../data/Physicians.csv")

print(f"\n✓ Loaded Physicians dataset: {df_physicians.shape}")
df_physicians.head()

In [ ]:
# Download Hospitals Dataset
print("Downloading Hospitals dataset...")
gdown.download(id="1X0TXbHc-5y1_W5mPSZM7SXgSVin1dSo9", output="../data/hospitals.csv", quiet=False)

df_hospitals = pd.read_csv("../data/hospitals.csv")

print(f"\n✓ Loaded Hospitals dataset: {df_hospitals.shape}")
df_hospitals.head()

In [ ]:
# Check data structure for both datasets
print("Physicians columns:", df_physicians.columns.tolist())
print("\nHospitals columns:", df_hospitals.columns.tolist())

# TODO: Calculate per 100k metrics once we have population data
print("\n⚠ Note: Will calculate Physicians_Per100k and Hospitals_Per100k after loading population data")

## 6. Environmental and Behavioral Risk Factors

Loading physical inactivity and PM2.5 pollution datasets.

In [ ]:
# Download Physical Inactivity Dataset
print("Downloading Physical Inactivity dataset...")
gdown.download(id="17CymbIRG6qrmcL-ZMYDqWTOB0c8PtaT4", output="../data/PhysicalActivity.csv", quiet=False)

df_inactivity = pd.read_csv("../data/PhysicalActivity.csv")

print(f"\n✓ Loaded Physical Inactivity dataset: {df_inactivity.shape}")
df_inactivity.head()

In [ ]:
# Download PM2.5 Pollution Dataset
print("Downloading PM2.5 Pollution dataset...")
gdown.download(id="1C1GpHWB8wjriN13Jw5O7NM8nqQOVN1AU", output="../data/pollution.csv", quiet=False)

df_pollution = pd.read_csv("../data/pollution.csv")

print(f"\n✓ Loaded PM2.5 Pollution dataset: {df_pollution.shape}")
df_pollution.head()

In [ ]:
# Download Population Dataset (needed for per-capita calculations)
print("Downloading Population dataset...")
gdown.download(id="1SiHB7Td9piKz3QpTQnDuEWCU7K6U-r6i", output="../data/Population.csv", quiet=False)

df_population = pd.read_csv("../data/Population.csv")

print(f"\n✓ Loaded Population dataset: {df_population.shape}")
df_population.head()

## 7. Summary and Next Steps

**Data Loading Complete:**
- ✅ Chronic Disease Dataset (cleaned)
- ✅ Healthcare Expenditure Dataset
- ✅ Physicians Dataset  
- ✅ Hospitals Dataset
- ✅ Physical Inactivity Dataset
- ✅ PM2.5 Pollution Dataset
- ✅ Population Dataset

**Next Steps:**
- Merge all datasets into master dataframe
- Engineer per-capita and per-100k features
- Conduct comprehensive EDA
- Build predictive ML models
- Perform clustering analysis
- Answer research questions

---

**Next Steps:**
- Load remaining 7 datasets
- Clean and preprocess each dataset
- Create master dataframe
- Perform EDA
- Build ML models